# 02 · Exploración de cinco años: clasificación ABC y XYZ global

Flujo: (1) cargar y revisar las fuentes, (2) calcular las métricas por artículo sobre los cinco años, (3) clasificar ABC y XYZ, (4) cruzar ABC × XYZ y (5) graficar. Cada cálculo se hace a la vista y su resultado se muestra en el notebook. No se congela nada.

In [ ]:
import os
import sys
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/. PROJECT_ROOT permite apuntar a otra carpeta
# de datos (por ejemplo, una copia sintética para validar el flujo).
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import PercentFormatter

from src.config import all_years, load_validated_config
from src.data import load_calendar, load_interim
from src.segmentation import (
    build_matrix,
    classify_abc,
    classify_xyz,
    compute_cv,
    compute_usage_spend,
    describe_items,
)

# Si es True, la figura de la sección 5 se guarda en reports/figures/.
SAVE_FIGURES = False

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)
%matplotlib inline

cfg = load_validated_config()
seg = cfg["segmentation"]
print("versión de la config:", cfg["version"])
print("segmentación:", seg)

## 1 · Fuentes de datos

Vista previa de los archivos crudos y de los arreglos intermedios que alimentan los cálculos.

In [ ]:
raw_dir = Path(cfg["data"]["raw_dir"])
last_day = cfg["data"]["last_day"]

# Calendario: un registro por día (d_1 … d_n) con su semana wm_yr_wk.
calendar = load_calendar(cfg)
print(f"calendar.csv: {calendar.shape[0]} días, de {calendar['date'].min()} a {calendar['date'].max()}")
display(calendar.head())

In [ ]:
# Ventas crudas (solo 5 filas y las primeras columnas de días): una fila por serie tienda-artículo.
sales_head = pd.read_csv(raw_dir / cfg["data"]["sales_file"], nrows=5)
print(f"{cfg['data']['sales_file']}: {sales_head.shape[1]} columnas en la vista previa")
display(sales_head.iloc[:, :12])

# Precios crudos (solo 5 filas): precio de venta por tienda, artículo y semana.
prices_head = pd.read_csv(raw_dir / "sell_prices.csv", nrows=5)
display(prices_head)

In [ ]:
# Intermedios por artículo (unidades y gasto = Σ unidades × precio por tienda-día), días 1..last_day.
# NaN = día sin dato (prelanzamiento o enmascarado).
source_data_artifact = load_interim(cfg, last_day)
item_ids = source_data_artifact["item_ids"]
print(f"artículos: {len(item_ids)} · días: {source_data_artifact['item_units'].shape[1]}")

day_cols = [f"d_{d}" for d in range(1, 9)]
display(pd.DataFrame(source_data_artifact["item_units"][:5, :8], index=item_ids[:5], columns=day_cols).rename_axis("unidades"))
display(pd.DataFrame(source_data_artifact["item_spend"][:5, :8], index=item_ids[:5], columns=day_cols).rename_axis("gasto"))
display(pd.DataFrame({"item_id": item_ids, "item_launch_d": source_data_artifact["item_launch_d"]}).head())

In [ ]:
# Ventana de los cinco años (los días anteriores a Y1 son historia previa y no se usan aquí).
years = all_years(cfg)
years_table = pd.DataFrame(years, index=["start_d", "end_d"]).T
years_table["days"] = years_table["end_d"] - years_table["start_d"] + 1
display(years_table)

start_d, end_d = years["Y1"][0], years["Y5"][1]
print(f"ventana global: d_{start_d} a d_{end_d} ({end_d - start_d + 1} días)")

## 2 · Métricas globales (los cinco años)

- **Uso** y **gasto**: suma de unidades y de Σ unidades × precio en la ventana; un artículo es *activo* si tiene al menos un día con dato.
- **CV**: desviación estándar poblacional / media × 100 sobre bloques completos de 28 días (`cv_aggregation = block28`); es indefinido con menos de `cv_min_blocks` bloques o media 0.
- **Descripción diaria**: media, mediana y proporción de ceros sobre días activos.

In [ ]:
# Uso y gasto por artículo.
usage_spend = compute_usage_spend(source_data_artifact["item_units"], source_data_artifact["item_spend"], start_d, end_d)
usage_spend.insert(0, "item_id", item_ids)
display(usage_spend.head())

# CV por artículo.
cv_df = compute_cv(
    source_data_artifact["item_units"], start_d, end_d, seg["cv_aggregation"], seg["cv_min_blocks"], source_data_artifact["week_of_day"]
)
display(cv_df.head())

# Media, mediana y proporción de ceros por día.
daily = describe_items(source_data_artifact, start_d, end_d)
display(daily.head())

In [ ]:
# Una sola tabla de métricas por artículo.
metrics = (
    usage_spend.rename(columns={"usage": "usage_units"})
    .join(cv_df)
    .merge(daily.drop(columns="total_units"), on="item_id")
)
display(metrics.head())

# Resumen global: lo que se agrega y lo que queda sin cálculo.
active = metrics[metrics["active"]]
summary = pd.Series(
    {
        "artículos": len(metrics),
        "activos en la ventana": len(active),
        "inactivos": int((~metrics["active"]).sum()),
        "activos con CV indefinido": int(active["cv"].isna().sum()),
        "unidades totales": metrics["usage_units"].sum(),
        "gasto total": metrics["spend"].sum(),
        "CV mediano (%)": active["cv"].median(),
    },
    name="valor",
)
display(summary.to_frame())
display(metrics[["usage_units", "spend", "cv", "n_blocks", "mean_daily_units", "median_daily_units", "zero_share"]].describe())

## 3 · Matriz de clasificación por artículo

- **ABC**: se ordena por gasto descendente y se mira el % acumulado *antes* de sumar el artículo; es A si es menor que el primer corte, B si es menor que el segundo y C en otro caso.
- **XYZ**: X si CV < umbral X, Y si CV < umbral Z y Z en otro caso.

In [ ]:
# ABC: solo artículos activos, ordenados por gasto (empates por item_id).
abc = classify_abc(active["item_id"], active["spend"].to_numpy(), seg["abc_cuts"])
display(abc.head())
display(abc.groupby("abc_class").agg(items=("item_id", "size"), spend=("spend", "sum")).assign(
    spend_pct=lambda t: t["spend"] / t["spend"].sum() * 100))

Pareto con doble eje y sensibilidad de los cortes ABC

Barras: gasto anual promedio por artículo (gasto de los cinco años / 5), en el eje principal. Línea: gasto acumulado porcentual, en el eje secundario. Las barras se colorean con los cortes configurados.

In [ ]:
n_years = len(years)
colors = {"A": "#1f77b4", "B": "#ff7f0e", "C": "#7f7f7f"}
# abc ya trae solo los artículos activos, ordenados por gasto, con cum_spend y abc_class.
ranked = abc.copy()
rank = np.arange(1, len(ranked) + 1)
annual_spend = ranked["spend"] / n_years
cum_pct = ranked["cum_spend"] / ranked["spend"].sum()

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.bar(rank, annual_spend, width=1.0, color=ranked["abc_class"].map(colors), linewidth=0)
ax.set(title="Pareto: gasto anual por artículo y gasto acumulado", xlabel="artículos activos, por gasto descendente",
       ylabel="gasto anual promedio por artículo")
ax.margins(x=0)

ax2 = ax.twinx()
ax2.plot(rank, cum_pct, color="black", linewidth=1.8)
for cut in seg["abc_cuts"]:
    ax2.axhline(cut, color="grey", linestyle="--", linewidth=1)
    ax2.annotate(f"{cut:.0%}", (rank[-1], cut), xytext=(-4, 4), textcoords="offset points", ha="right")
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
ax2.set(ylabel="gasto acumulado", ylim=(0, 1.02))

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
ax.legend(handles, [f"{k} ({(ranked['abc_class'] == k).sum()})" for k in colors], title="clase (artículos)", loc="center right")
fig.tight_layout()
if SAVE_FIGURES:
    out = Path("reports/figures")
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / "eda_five_years_pareto.png", dpi=150, bbox_inches="tight")
    print("figura guardada en", out / "eda_five_years_pareto.png")
plt.show()

### Escenarios de corte

Se repite `classify_abc` con distintos cortes de gasto acumulado (A, B). Por cada escenario: recuento de artículos, % de los artículos (unidades de inventario, SKU) y % del gasto de cada clase.

In [ ]:
a_cuts = [0.50, 0.60, 0.70, 0.80, 0.90]
b_cuts = [0.90, 0.95, 0.98]
total_items = len(ranked)

rows = []
for a in a_cuts:
    for b in b_cuts:
        if a >= b:
            continue
        r = classify_abc(ranked["item_id"], ranked["spend"].to_numpy(), [a, b])
        by = r.groupby("abc_class").agg(items=("item_id", "size"), spend=("spend", "sum"))
        by = by.reindex(["A", "B", "C"], fill_value=0)
        row = {"corte_A": a, "corte_B": b}
        for cls in ("A", "B", "C"):
            row[f"{cls}_items"] = int(by.loc[cls, "items"])
            row[f"{cls}_%items"] = by.loc[cls, "items"] / total_items * 100
            row[f"{cls}_%gasto"] = by.loc[cls, "spend"] / by["spend"].sum() * 100
        rows.append(row)
scenarios = pd.DataFrame(rows)
scenarios["escenario"] = [f"A≤{a:.0%} · B≤{b:.0%}" for a, b in zip(scenarios["corte_A"], scenarios["corte_B"])]
assert (scenarios[["A_items", "B_items", "C_items"]].sum(axis=1) == total_items).all()
display(scenarios.set_index("escenario").round(2))

In [ ]:
# Reparto de artículos y de gasto por escenario (el escenario configurado va marcado con *).
labels = [("* " if [a, b] == list(seg["abc_cuts"]) else "") + e
          for a, b, e in zip(scenarios["corte_A"], scenarios["corte_B"], scenarios["escenario"])]
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, kind, title in zip(axes, ("%items", "%gasto"), ("% de artículos (SKU)", "% del gasto")):
    left = np.zeros(len(scenarios))
    for cls, color in colors.items():
        vals = scenarios[f"{cls}_{kind}"].to_numpy()
        ax.barh(labels, vals, left=left, color=color, label=cls)
        for y, (l, v) in enumerate(zip(left, vals)):
            if v >= 4:
                ax.text(l + v / 2, y, f"{v:.0f}%", ha="center", va="center", color="white", fontsize=8)
        left += vals
    ax.set(title=title, xlim=(0, 100))
    ax.xaxis.set_major_formatter(PercentFormatter(100))
axes[0].invert_yaxis()
axes[1].legend(title="clase", loc="upper center", bbox_to_anchor=(-0.1, -0.06), ncol=3)
fig.tight_layout()
plt.show()

In [ ]:
raise Exceptionq("Detener")

In [ ]:
# XYZ: sobre el CV de los artículos activos.
xyz = classify_xyz(active["cv"], seg["xyz_thresholds"])
display(xyz.value_counts(dropna=False).rename("items").to_frame())

In [ ]:
# Matriz final: métricas + clase ABC + clase XYZ por artículo (inactivos al final, sin clase).
matrix = (
    metrics[["item_id", "active", "usage_units", "spend", "cv", "n_blocks"]]
    .merge(abc[["item_id", "cum_spend", "cum_pct_before", "abc_class"]], on="item_id", how="left")
    .merge(
        pd.DataFrame({"item_id": active["item_id"].to_numpy(), "xyz_class": xyz.to_numpy()}),
        on="item_id",
        how="left",
    )
    .sort_values(["active", "spend", "item_id"], ascending=[False, False, True])
    .reset_index(drop=True)
)
matrix["cell"] = np.where(matrix["active"], matrix["abc_class"].fillna("") + matrix["xyz_class"].fillna("-"), None)
display(matrix.head(10))
display(matrix.tail(5))

# Control: el mismo resultado debe salir de build_matrix (la función que usa el pipeline).
ref = build_matrix("check", source_data_artifact, start_d, end_d, seg, source_data_artifact["week_of_day"]).set_index("item_id")
mine = matrix.set_index("item_id").loc[ref.index]
same_abc = (mine["abc_class"].fillna("-") == ref["abc_class"].fillna("-")).all()
same_xyz = (mine["xyz_class"].fillna("-") == ref["xyz_class"].fillna("-")).all()
print("coincide con build_matrix · ABC:", same_abc, "· XYZ:", same_xyz)

## 4 · Matriz cruzada ABC × XYZ

Recuento de artículos. `-` en XYZ = CV indefinido; `inactivo` en ABC = sin ventas en la ventana.

In [ ]:
abc_lbl = matrix["abc_class"].fillna("inactivo")
xyz_lbl = matrix["xyz_class"].fillna("-")

counts = pd.crosstab(abc_lbl, xyz_lbl, margins=True, margins_name="Total")
assert counts.loc["Total", "Total"] == len(matrix), "la tabla debe sumar todos los artículos"
print(f"total de artículos en la tabla: {counts.loc['Total', 'Total']}")
display(counts)

In [ ]:
# % sobre el total de cada fila (cómo se reparte cada clase ABC entre X, Y y Z).
by_row = pd.crosstab(abc_lbl, xyz_lbl, normalize="index", margins=True, margins_name="Total") * 100
print("% por fila")
display(by_row.round(2))

# % sobre el total de cada columna (cómo se reparte cada clase XYZ entre A, B y C).
by_col = pd.crosstab(abc_lbl, xyz_lbl, normalize="columns", margins=True, margins_name="Total") * 100
print("% por columna")
display(by_col.round(2))

## 5 · Gráficas

Pareto de la curva ABC, CV en escala logarítmica con los umbrales XYZ y media frente a mediana diaria.

In [ ]:
colors = {"A": "#1f77b4", "B": "#ff7f0e", "C": "#7f7f7f"}
ranked = matrix[matrix["active"]].reset_index(drop=True)
rank_pct = (np.arange(1, len(ranked) + 1)) / len(ranked)
cum_pct = ranked["cum_spend"] / ranked["spend"].sum()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pareto: gasto acumulado frente a % de artículos, coloreado por clase ABC.
ax = axes[0]
for cls, color in colors.items():
    mask = (ranked["abc_class"] == cls).to_numpy()
    ax.plot(rank_pct[mask], cum_pct[mask], color=color, linewidth=2.5, label=f"{cls} ({mask.sum()})")
for cut in seg["abc_cuts"]:
    ax.axhline(cut, color="grey", linestyle="--", linewidth=1)
    ax.annotate(f"{cut:.0%}", (1.0, cut), xytext=(-4, 4), textcoords="offset points", ha="right")
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set(title="Pareto del gasto (5 años)", xlabel="artículos activos, por gasto descendente",
       ylabel="gasto acumulado")
ax.legend(title="clase (artículos)", loc="lower right")

# CV en escala logarítmica con los umbrales XYZ.
ax = axes[1]
cv = ranked["cv"].dropna()
cv = cv[cv > 0]
ax.hist(cv, bins=np.logspace(np.log10(cv.min()), np.log10(cv.max()), 60), color="#4c78a8")
ax.set_xscale("log")
for thr, label in zip(seg["xyz_thresholds"], ("X|Y", "Y|Z"), strict=True):
    ax.axvline(thr, color="crimson", linestyle="--", linewidth=1.2)
    ax.annotate(f"{label}: {thr:g}%", (thr, 1.0), xycoords=("data", "axes fraction"),
                xytext=(4, -14), textcoords="offset points", color="crimson")
ax.set(title="CV de unidades por artículo", xlabel="CV (%, escala log)", ylabel="artículos")

# Media frente a mediana diaria: sobre la diagonal y = x la media supera a la mediana.
ax = axes[2]
d = matrix[["item_id"]].merge(daily, on="item_id")
d = d[d["active_days"] > 0]
ax.scatter(d["median_daily_units"], d["mean_daily_units"], s=5, alpha=0.3, color="#4c78a8")
top = np.nanmax(d[["median_daily_units", "mean_daily_units"]].to_numpy())
ax.plot([0, top], [0, top], color="grey", linestyle="--", linewidth=1, label="media = mediana")
ax.set_xscale("symlog")
ax.set_yscale("symlog")
ax.set(title="Media frente a mediana diaria", xlabel="mediana (unidades/día)", ylabel="media (unidades/día)")
ax.legend(loc="upper left")

fig.tight_layout()
if SAVE_FIGURES:
    out = Path("reports/figures")
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / "eda_five_years_abc_xyz.png", dpi=150, bbox_inches="tight")
    print("figura guardada en", out / "eda_five_years_abc_xyz.png")
plt.show()